# 07 — BERT Sentiment Classification
**Objective 6:** Fine-tune a pre-trained BERT model for Positive / Neutral / Negative classification.

> On CPU, use a sample (`SAMPLE_SIZE = 8000`) and fewer epochs.  
> Requires: `transformers`, `torch`, `accelerate`, `datasets`.


In [ ]:
# %pip install transformers torch accelerate datasets scikit-learn

from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "outputs" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

SENTIMENT_LABELS = ["negative", "neutral", "positive"]
LABEL2ID = {l: i for i, l in enumerate(SENTIMENT_LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
RANDOM_SEED = 42
TEST_SIZE, VAL_SIZE = 0.15, 0.15

# Hyper-parameters
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 128
BATCH_SIZE = 16          # lower if OOM
LEARNING_RATE = 2e-5
EPOCHS = 2               # increase to 3–4 for full runs
SAMPLE_SIZE = 8000       # set None for full data

def load_cleaned():
    for name in ("cleaned_reviews.parquet", "cleaned_reviews.pkl"):
        p = DATA_DIR / name
        if p.exists() and p.stat().st_size > 0:
            return pd.read_parquet(p) if name.endswith(".parquet") else pd.read_pickle(p)
    raise FileNotFoundError("Run 02_data_cleaning first")

df = load_cleaned()
if SAMPLE_SIZE and SAMPLE_SIZE < len(df):
    df = df.sample(SAMPLE_SIZE, random_state=RANDOM_SEED)
    print(f"Using sample of {len(df)} reviews")
else:
    print(f"Using all {len(df)} reviews")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
    }

texts = df["review_text_clean"].tolist()
labels = df["sentiment_label"].map(LABEL2ID).tolist()

X_temp, X_test, y_temp, y_test = train_test_split(
    texts, labels, test_size=TEST_SIZE, stratify=labels, random_state=RANDOM_SEED
)
val_ratio = VAL_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_ratio, stratify=y_temp, random_state=RANDOM_SEED
)
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(SENTIMENT_LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

train_ds = ReviewDataset(X_train, y_train, tokenizer, MAX_LENGTH)
val_ds   = ReviewDataset(X_val,   y_val,   tokenizer, MAX_LENGTH)
test_ds  = ReviewDataset(X_test,  y_test,  tokenizer, MAX_LENGTH)

output_dir = MODELS_DIR / "bert_sentiment"

training_args = TrainingArguments(
    output_dir=str(output_dir),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    seed=RANDOM_SEED,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
print("Trainer ready.")


In [ ]:
# Fine-tune (this cell takes time)
train_result = trainer.train()
print("Training metrics:", train_result.metrics)


In [ ]:
# Evaluate on test set
test_output = trainer.predict(test_ds)
test_preds = np.argmax(test_output.predictions, axis=-1)

metrics = {
    "accuracy": float(accuracy_score(y_test, test_preds)),
    "f1_macro": float(f1_score(y_test, test_preds, average="macro", zero_division=0)),
    "precision_macro": float(precision_score(y_test, test_preds, average="macro", zero_division=0)),
    "recall_macro": float(recall_score(y_test, test_preds, average="macro", zero_division=0)),
}
print("Test metrics:", metrics)
print(classification_report(y_test, test_preds, target_names=SENTIMENT_LABELS, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, test_preds))

# Save
trainer.save_model(str(output_dir))
tokenizer.save_pretrained(str(output_dir))
pd.Series(metrics).to_csv(MODELS_DIR / "bert_test_metrics.csv")
print(f"Model saved → {output_dir}")
